# SMART — Ablations, Sweeps and Statistical AnalysisEverything in this notebook calls the same code that `evaluate.py` calls, so anynumber produced here is reproducible from the command line. Nothing isre-implemented.Contents:1. Setup and regression check2. Component ablation (row centering / CDF / temporal edges)3. Propagation ablation (independent per-class walks vs joint diffusion)4. Feature ablation (backbone and modality)5. Window-size (N) and RW-step sweeps6. Graph-pruning sweep (CDF gamma vs kNN k)7. Decoder comparison (F-beta thresholds vs rank-gap)8. Seed-selection variance and significance testing9. Qualitative timeline figure for the paperRun `dataset.py`, `train.py` and `extract_features.py` first — this notebookreads only the artifacts they produce.

## 1. Setup and regression checkSet `DATA_DIR`, `UNITS` and `FEAT_SUFFIX` to your run, then confirm the FULLconfiguration reproduces your locked headline number **before** changinganything. If it does not, stop — every table below inherits the error.

In [ ]:
import sys, pickle, itertoolsfrom pathlib import Pathimport numpy as npsys.path.insert(0, '..')          # notebook lives in tools/, code lives at the repo rootfrom evaluate import Config, run_all, run_unit, run_pooled, timeline_figure, load_unitfrom tools.metrics import ranking_metrics# ---- edit these three lines --------------------------------------------DATA_DIR    = Path('../artifacts/adl')      # or ../artifacts/charadesUNITS       = ['P_09','P_10','P_11','P_12','P_16','P_17']FEAT_SUFFIX = '_resnet'                     # '' | _resnet | _effnet | _clip | _rgbonly | _flowonly# ------------------------------------------------------------------------N_WINDOW = 100        # ADL 100 (150 for the headline table); Charades 60RW_STEPS = 10CFG = Config(rw_steps=RW_STEPS)print(CFG)

In [ ]:
# REGRESSION CHECK — this must reproduce your locked FULL mAP exactly.base = run_all(DATA_DIR, UNITS, CFG, FEAT_SUFFIX, window=N_WINDOW, verbose=False)BASE_MAP = base['mean']['mAP']print('FULL config mean metrics:')for k, v in base['mean'].items():    print(f'  {k:<14}{v:.4f}')print(f'\nlocked headline mAP = {BASE_MAP:.4f}')

## 2. Component ablationToggles one head/graph component at a time. `dmAP` is the change relative to theFULL configuration — a large negative number means the component matters.

In [ ]:
from dataclasses import replacedef variants_table(data_dir, units, feat_suffix, N, steps, extra=None):    configs = {        'FULL (all on)'        : dict(),        '- row centering'      : dict(row_center=False),        '- CDF sparsification' : dict(use_cdf=False),        '- temporal edges'     : dict(use_temporal=False),        '- co-occurrence'      : dict(lam=0.0),        '- temporal smoothing' : dict(w_smooth=1),        '- background gate'    : dict(act_pctl=None),        '- max_k cap'          : dict(max_k=None),    }    if extra: configs.update(extra)    keys = ['mAP','macro_f1','micro_f1','top1']    res = {}    for name, over in configs.items():        cfg = Config(rw_steps=steps, **over)        res[name] = run_all(data_dir, units, cfg, feat_suffix, window=N, verbose=False)['mean']    full = res['FULL (all on)']    print(f'COMPONENT ABLATION (N={N}, steps={steps}, {len(units)} units)\n')    print(f'{"config":<24}' + ''.join(f'{k:>11}' for k in keys) + f'{"dmAP":>10}')    print('-'*(24+11*len(keys)+10))    for name in configs:        r = res[name]        print(f'{name:<24}' + ''.join(f'{r[k]:>11.4f}' for k in keys)              + f'{r["mAP"]-full["mAP"]:>+10.4f}')    return resABL_COMPONENTS = variants_table(DATA_DIR, UNITS, FEAT_SUFFIX, N_WINDOW, RW_STEPS)

## 3. Propagation ablation — independent vs jointThe single design choice that makes multi-action prediction possible. In`independent` mode each class runs its own random walk and never competes; in`joint` mode all classes diffuse over the same graph and are renormalised acrossclasses at every step, so mass gained by one class is taken from the others.Expected pattern: joint propagation suppresses co-occurring actions, so mAP andmacro-F1 drop sharply while top-1 (the single dominant action) barely moves.

In [ ]:
def compare_propagation(data_dir, units, feat_suffix, N, steps):    keys = ['mAP','macro_f1','micro_f1','top1']    out = {}    for mode in ['independent','joint']:        cfg = Config(rw_steps=steps, prop_mode=mode)        out[mode] = run_all(data_dir, units, cfg, feat_suffix, window=N, verbose=False)['mean']    print(f'PROPAGATION ABLATION (N={N}, steps={steps}, {len(units)} units)\n')    print(f'{"mode":<24}' + ''.join(f'{k:>11}' for k in keys))    print('-'*(24+11*len(keys)))    for mode, m in out.items():        tag = 'independent (SMART)' if mode=='independent' else 'joint (classical LP)'        print(f'{tag:<24}' + ''.join(f'{m[k]:>11.4f}' for k in keys))    d = out['independent']['mAP'] - out['joint']['mAP']    print(f'\nindependence gains {d:+.4f} mAP')    return outABL_PROP = compare_propagation(DATA_DIR, UNITS, FEAT_SUFFIX, N_WINDOW, RW_STEPS)

## 4. Feature ablation — backbone and modalityThese require re-running `extract_features.py` once per configuration, eachwriting a different `--feat_suffix` into the same artifacts folder. Only thesuffixes that actually exist on disk are evaluated.```bashpython train.py --backbone resnet50        ... --save_path checkpoints/r50.ptpython extract_features.py --checkpoint checkpoints/r50.pt ...   #  -> *_feats_resnet.npzpython train.py --backbone efficientnet_b0 ... --save_path checkpoints/eff.ptpython extract_features.py --checkpoint checkpoints/eff.pt ...   #  -> *_feats_effnet.npzpython train.py --backbone clip            ... --save_path checkpoints/clip.ptpython extract_features.py --checkpoint checkpoints/clip.pt ...  #  -> *_feats_clip.npzpython train.py --backbone clip --modality rgbonly  ... --save_path checkpoints/rgb.ptpython extract_features.py --checkpoint checkpoints/rgb.pt  ...  #  -> *_feats_rgbonly.npzpython train.py --backbone clip --modality flowonly ... --save_path checkpoints/flow.ptpython extract_features.py --checkpoint checkpoints/flow.pt ...  #  -> *_feats_flowonly.npz```

In [ ]:
FEATURE_SETS = {    'ResNet-18 + flow (fusion)'  : '',    'ResNet-50 + flow (fusion)'  : '_resnet',    'EfficientNet-B0 + flow'     : '_effnet',    'CLIP ViT-B/32 + flow'       : '_clip',    'CLIP RGB only (no fusion)'  : '_rgbonly',    'CLIP flow only (no DEFT)'   : '_flowonly',}def feature_ablation(data_dir, units, N, steps, feature_sets=FEATURE_SETS):    keys = ['mAP','macro_f1','micro_f1','top1']    rows = {}    for name, suf in feature_sets.items():        missing = [u for u in units if not (Path(data_dir)/f'{u}_feats{suf}.npz').exists()]        if missing:            print(f'skip "{name}" (suffix "{suf}") — features missing for {missing[:3]}')            continue        cfg = Config(rw_steps=steps)        rows[name] = run_all(data_dir, units, cfg, suf, window=N, verbose=False)['mean']    if not rows:        print('no feature sets found — run extract_features.py first'); return {}    print(f'\nFEATURE ABLATION (N={N}, steps={steps}, {len(units)} units)\n')    print(f'{"features":<30}' + ''.join(f'{k:>11}' for k in keys))    print('-'*(30+11*len(keys)))    for name, m in rows.items():        print(f'{name:<30}' + ''.join(f'{m[k]:>11.4f}' for k in keys))    return rowsABL_FEATURES = feature_ablation(DATA_DIR, UNITS, N_WINDOW, RW_STEPS)

## 5. Window size (N) and RW stepsN is the main in-pipeline tuning lever: it sets how many frames share one graph.Too small and there is no context to propagate through, too large and the windowspans unrelated activity.

In [ ]:
def sweep_N(data_dir, units, feat_suffix, Ns=(30,60,100,150,200), steps=RW_STEPS):    keys = ['mAP','macro_f1','micro_f1','top1']    print(f'WINDOW-SIZE SWEEP (steps={steps}, {len(units)} units)\n')    print(f'{"N":>6}' + ''.join(f'{k:>11}' for k in keys))    print('-'*(6+11*len(keys)))    out = {}    for N in Ns:        m = run_all(data_dir, units, Config(rw_steps=steps), feat_suffix,                    window=N, verbose=False)['mean']        out[N] = m        print(f'{N:>6}' + ''.join(f'{m[k]:>11.4f}' for k in keys))    best = max(out, key=lambda n: out[n]['mAP'])    print(f'\nbest N = {best} (mAP {out[best]["mAP"]:.4f})')    return outSWEEP_N = sweep_N(DATA_DIR, UNITS, FEAT_SUFFIX)

In [ ]:
def sweep_steps(data_dir, units, feat_suffix, N=N_WINDOW, step_list=(1,2,3,5,10,20)):    keys = ['mAP','macro_f1','top1']    print(f'RW-STEP SWEEP (N={N}, {len(units)} units)\n')    print(f'{"steps":>6}' + ''.join(f'{k:>11}' for k in keys))    print('-'*(6+11*len(keys)))    out = {}    for t in step_list:        m = run_all(data_dir, units, Config(rw_steps=t), feat_suffix,                    window=N, verbose=False)['mean']        out[t] = m        print(f'{t:>6}' + ''.join(f'{m[k]:>11.4f}' for k in keys))    return outSWEEP_STEPS = sweep_steps(DATA_DIR, UNITS, FEAT_SUFFIX)

## 6. Graph-pruning sweep — CDF gamma vs kNN kBoth rules prune the same dense affinity matrix and everything downstream isidentical, so the comparison isolates the pruning rule alone. CDF adapts theneighbourhood size per frame; kNN fixes it.

In [ ]:
def pruning_sweep(data_dir, units, feat_suffix, N=N_WINDOW, steps=RW_STEPS,                  ks=(5,10,15,20), gammas=(0.80,0.85,0.90,0.95)):    rows = []    for k in ks:        m = run_all(data_dir, units, Config(rw_steps=steps, sparsify='knn', knn_k=k),                    feat_suffix, window=N, verbose=False)['mean']        rows.append(('kNN', k, m))    for g in gammas:        m = run_all(data_dir, units, Config(rw_steps=steps, sparsify='cdf', gamma=g),                    feat_suffix, window=N, verbose=False)['mean']        rows.append(('CDF', g, m))    print(f'PRUNING SWEEP (N={N}, steps={steps}, {len(units)} units)\n')    print(f'{"rule":<8}{"value":>8}{"mAP":>10}{"macroF1":>10}{"microF1":>10}{"top1":>10}')    print('-'*56)    for meth, val, m in rows:        print(f'{meth:<8}{val:>8}{m["mAP"]:>10.4f}{m["macro_f1"]:>10.4f}'              f'{m["micro_f1"]:>10.4f}{m["top1"]:>10.4f}')    return rowsSWEEP_PRUNE = pruning_sweep(DATA_DIR, UNITS, FEAT_SUFFIX)

## 7. Decoder comparison — F-beta thresholds vs rank-gap**Sanity condition:** mAP and top-1 are computed from the scores, not the decodedsets, so they must be *identical* across the two rows. Only macro/micro-F1 candiffer. If mAP moves, something is wrong — stop and debug before reporting.

In [ ]:
def compare_decoders(data_dir, units, feat_suffix, N=N_WINDOW, steps=RW_STEPS):    keys = ['mAP','macro_f1','micro_f1','top1']    out = {}    for dec in ['fbeta','rankgap']:        m = run_all(data_dir, units, Config(rw_steps=steps, decoder=dec),                    feat_suffix, window=N, verbose=False)['mean']        out[dec] = m    print(f'DECODER COMPARISON (N={N}, steps={steps}, {len(units)} units)\n')    print(f'{"decoder":<12}' + ''.join(f'{k:>11}' for k in keys))    print('-'*(12+11*len(keys)))    for dec, m in out.items():        print(f'{dec:<12}' + ''.join(f'{m[k]:>11.4f}' for k in keys))    same = abs(out['fbeta']['mAP'] - out['rankgap']['mAP']) < 1e-9    print(f'\nmAP identical across decoders: {same}  <-- must be True')    return outDECODERS = compare_decoders(DATA_DIR, UNITS, FEAT_SUFFIX)

## 8. Seed-selection variance and significanceThe seed mask is a random draw of ~10% of the labelled frames, so a single numberis a single draw. This re-draws the seeds `n_runs` times with different RNG seedsand reports mean +/- std and a 95% CI, then tests whether row centering helpssignificantly (paired across units and across runs).

In [ ]:
from tools.graph import make_windows, build_window_graph, propagate_dispatchfrom tools.head import cooccurrence, head_transform, fit_thresholds, h75_thresholdfrom tools.metrics import evaluate as eval_metrics, topk_accuracyfrom dataset import stratified_seedsdef run_units_reseeded(data_dir, units, feat_suffix, run_seed, cfg,                       N=N_WINDOW, seed_frac=0.10, min_per_class=2):    """Re-draw the seed mask with `run_seed` and rerun the whole method."""    per = []    for u in units:        feats, Y, _, _ = load_unit(data_dir, u, feat_suffix)        n, C = Y.shape        seeds = stratified_seeds(Y, seed_frac, min_per_class,                                 np.random.default_rng(run_seed))        Co = cooccurrence(Y[seeds])        S = np.zeros((n, C)); covered = np.zeros(n, bool)        for w in make_windows(n, N):            idx = np.array(w, int)            P, _ = build_window_graph(feats[idx], cfg.feat_norm, cfg.sigma_mode,                                      cfg.use_cdf, cfg.use_temporal,                                      cfg.sparsify, cfg.gamma, cfg.knn_k)            raw = propagate_dispatch(P, Y[idx], seeds[idx], cfg.rw_steps, cfg.prop_mode)            S[idx] = head_transform(raw, P, Co, cfg.w_smooth, cfg.lam, cfg.row_center)            covered[idx] = True        taus = fit_thresholds(S[seeds], Y[seeds], cfg.fbeta)        act = (np.percentile(S[seeds].max(1), cfg.act_pctl)               if cfg.act_pctl is not None else None)        ev = covered & (~seeds)        pred = h75_threshold(S[ev], taus, cfg.topk, cfg.topk_floor, cfg.max_k, act)        m = eval_metrics(pred, S[ev], Y[ev]); m.update(topk_accuracy(S[ev], Y[ev], ks=(1,5)))        per.append({k: m[k] for k in ['mAP','macro_f1','micro_f1','top1']})    mean = {k: float(np.mean([p[k] for p in per])) for k in per[0]}    return per, mean

In [ ]:
def stats_report(data_dir, units, feat_suffix, n_runs=5, N=N_WINDOW,                 steps=RW_STEPS, base_seed=1000):    from scipy import stats as st    keys = ['mAP','macro_f1','micro_f1','top1']    full, norc = [], []    for r in range(n_runs):        _, fm = run_units_reseeded(data_dir, units, feat_suffix, base_seed+r,                                   Config(rw_steps=steps, row_center=True), N)        _, nm = run_units_reseeded(data_dir, units, feat_suffix, base_seed+r,                                   Config(rw_steps=steps, row_center=False), N)        full.append(fm); norc.append(nm)    def summ(runs):        return {k: (float(np.mean([x[k] for x in runs])),                    float(np.std([x[k] for x in runs], ddof=1))) for k in keys}    F, Nr = summ(full), summ(norc)    ci = 1.96/np.sqrt(n_runs)    print(f'(1) SEED-SELECTION VARIANCE over {n_runs} runs (N={N}, steps={steps})\n')    print(f'{"metric":<12}{"FULL  mean +/- std [95% CI]":<40}{"w/o row centering":>22}')    print('-'*74)    for k in keys:        fm, fs = F[k]; nm, ns = Nr[k]        print(f'{k:<12}{fm:.4f} +/- {fs:.4f}  [{fm-ci*fs:.4f}, {fm+ci*fs:.4f}]  '              f'{nm:>14.4f} +/- {ns:.4f}')    # paired test across units on one reference draw    fper, _ = run_units_reseeded(data_dir, units, feat_suffix, base_seed,                                 Config(rw_steps=steps, row_center=True), N)    nper, _ = run_units_reseeded(data_dir, units, feat_suffix, base_seed,                                 Config(rw_steps=steps, row_center=False), N)    fa = [p['mAP'] for p in fper]; na = [p['mAP'] for p in nper]    print(f'\n(2) Paired significance of row centering across {len(units)} units (mAP):')    try:        w, p = st.wilcoxon(fa, na)        print(f'    Wilcoxon signed-rank: W={w:.3f}, p={p:.5f}'              f' {"(significant)" if p < 0.05 else "(n.s.)"}')    except ValueError as e:        print('    Wilcoxon:', e)    t, pt = st.ttest_rel([x['mAP'] for x in full], [x['mAP'] for x in norc])    print(f'    Paired t-test across {n_runs} runs: t={t:.3f}, p={pt:.2e}'          f' {"(significant)" if pt < 0.05 else "(n.s.)"}')    return dict(full=full, no_row_center=norc)# STATS = stats_report(DATA_DIR, UNITS, FEAT_SUFFIX, n_runs=5)

## 9. Seed-ratio sweepHow much supervision does SMART actually need? Re-draws the seed mask at severalratios using the same machinery as section 8.

In [ ]:
def seed_ratio_sweep(data_dir, units, feat_suffix, fracs=(0.02,0.05,0.10,0.25,0.50),                     N=N_WINDOW, steps=RW_STEPS, seed=1000):    keys = ['mAP','macro_f1','micro_f1','top1']    print(f'SEED-RATIO SWEEP (N={N}, steps={steps}, {len(units)} units)\n')    print(f'{"seed %":>8}' + ''.join(f'{k:>11}' for k in keys))    print('-'*(8+11*len(keys)))    out = {}    for f in fracs:        _, m = run_units_reseeded(data_dir, units, feat_suffix, seed,                                  Config(rw_steps=steps), N, seed_frac=f)        out[f] = m        print(f'{100*f:>7.0f}%' + ''.join(f'{m[k]:>11.4f}' for k in keys))    return out# SEED_SWEEP = seed_ratio_sweep(DATA_DIR, UNITS, FEAT_SUFFIX)

## 10. Qualitative timeline figureGround truth (dark grey, upper bar) against SMART (blue, lower bar) for everyclass present in the unit. Vector output stays sharp at any LaTeX scale; coloursare Okabe-Ito, so the figure survives greyscale printing.If a unit has too many classes to render legibly, pass an explicit`classes=[...]` subset.

In [ ]:
%matplotlib inlineSHOWCASE = UNITS[0]m = run_unit(DATA_DIR, SHOWCASE, Config(rw_steps=RW_STEPS), FEAT_SUFFIX,             window=N_WINDOW, verbose=True)fig = timeline_figure(DATA_DIR, SHOWCASE, m,                      save_path=f'../images/timeline_{SHOWCASE}',                      formats=('png','pdf','svg'))

In [ ]:
# Multi-action examples: frames where SMART predicted two or more concurrent actionsnames = pickle.load(open(Path(DATA_DIR)/'class_map.pkl','rb'))['names']frames, pred, Yt = m['_frames'], m['_pred'], m['_Y']multi = [r for r in range(len(frames)) if pred[r].sum() >= 2]print(f'{SHOWCASE}: {len(multi)}/{len(frames)} frames predicted with >=2 actions'      f' ({100*len(multi)/max(len(frames),1):.1f}%)\n')for r in multi[:15]:    p = [names[c] for c in np.where(pred[r]==1)[0]]    t = [names[c] for c in np.where(Yt[r]==1)[0]]    tag = 'OK' if set(p) & set(t) else '--'    print(f'  frame {frames[r]:>6}: pred={p}  true={t}  [{tag}]')